<style>
:root {
  --ink: #17202a;
  --muted: #64748b;
  --paper: #fbf7ef;
  --card: #fffaf0;
  --line: #e7dcc9;
  --navy: #11263f;
  --teal: #0f766e;
  --amber: #b45309;
  --rose: #be123c;
}
body, .jp-Notebook { background: var(--paper); color: var(--ink); }
.jp-Cell, .cell { margin-bottom: 1.35rem; }
h1, h2, h3 { color: var(--navy); letter-spacing: -0.03em; }
h1 { font-size: 2.4rem; margin-bottom: .35rem; }
h2 { font-size: 1.55rem; margin-top: 2rem; }
p, li { font-size: 1.02rem; line-height: 1.65; }
a { color: var(--teal); font-weight: 700; }
.jano-hero { background: linear-gradient(135deg, #fff7ed 0%, #f0fdfa 54%, #e0f2fe 100%); border: 1px solid var(--line); border-radius: 28px; padding: 2rem; box-shadow: 0 22px 70px rgba(17, 38, 63, .10); }
.jano-eyebrow { color: var(--amber); text-transform: uppercase; font-size: .75rem; font-weight: 800; letter-spacing: .16em; }
.jano-lead { color: #334155; max-width: 900px; font-size: 1.08rem; }
.jano-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(170px, 1fr)); gap: 14px; margin: 1rem 0 1.25rem; }
.jano-card { background: var(--card); border: 1px solid var(--line); border-radius: 18px; padding: 1rem; box-shadow: 0 10px 30px rgba(17, 38, 63, .07); }
.jano-card strong { display: block; color: var(--muted); font-size: .72rem; text-transform: uppercase; letter-spacing: .12em; margin-bottom: .35rem; }
.jano-card span { color: var(--navy); font-size: 1.35rem; font-weight: 800; }
.jano-note { border-left: 5px solid var(--teal); background: #ecfeff; border-radius: 16px; padding: 1rem 1.2rem; color: #164e63; }
.jano-warning { border-left-color: var(--amber); background: #fffbeb; color: #78350f; }
.jano-section { background: rgba(255,255,255,.58); border: 1px solid var(--line); border-radius: 24px; padding: 1.25rem 1.35rem; margin: 1.2rem 0; }
.jano-table table, table.dataframe { border-collapse: collapse; width: 100%; font-size: .9rem; background: white; border-radius: 16px; overflow: hidden; box-shadow: 0 12px 35px rgba(17, 38, 63, .08); }
.jano-table th, .jano-table td, table.dataframe th, table.dataframe td { border-bottom: 1px solid #edf2f7; padding: .68rem .72rem; text-align: right; }
.jano-table th, table.dataframe th { background: #11263f; color: #f8fafc; font-weight: 700; }
.jano-table tr:nth-child(even) td, table.dataframe tr:nth-child(even) td { background: #f8fafc; }
.output_area, .jp-OutputArea-output { margin-top: .65rem; }
pre, code { font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace; }
pre.jano-code { background: #0f172a; color: #d1fae5; border-radius: 18px; padding: 1rem; overflow-x: auto; font-size: .83rem; line-height: 1.55; border: 1px solid #1e293b; }
details.jano-code-wrap { margin: .65rem 0; }
details.jano-code-wrap summary { cursor: pointer; color: var(--teal); font-weight: 800; font-size: .9rem; }
.jano-pill { display: inline-flex; align-items: center; gap: .35rem; border-radius: 999px; background: #ccfbf1; color: #134e4a; padding: .32rem .7rem; font-weight: 800; font-size: .82rem; margin: .2rem .25rem .2rem 0; }
.jano-pill.amber { background: #fef3c7; color: #92400e; }
.jano-pill.rose { background: #ffe4e6; color: #9f1239; }
</style>
<div class="jano-hero">
  <div class="jano-eyebrow">Real dataset walkthrough</div>
  <h1>Jano on BTS Airline On-Time Performance</h1>
  <p class="jano-lead">This notebook uses a real January 2024 extract from the U.S. Bureau of Transportation Statistics to show how Jano can inspect a leakage-aware temporal plan, execute a multiclass model over those folds, and compare retraining policies with an ordinal cost.</p>
  <span class="jano-pill">walk-forward plan</span>
  <span class="jano-pill amber">multiclass target</span>
  <span class="jano-pill rose">retrain runner</span>
</div>
<p>Official source:</p>
<ul>
<li>BTS On-Time Performance overview: https://www.transtats.bts.gov/ontime/</li>
<li>BTS table download page: https://www.transtats.bts.gov/DL_SelectFields.aspx?QO_fu146_anzr=b0-gvzr&amp;gnoyr_VQ=FGJ</li>
<li>Direct January 2024 PREZIP file used here: https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip</li>
</ul>
<div class="jano-note jano-warning">The BTS zip/CSV is intentionally kept out of git because it is a large external dataset. The repository ignores <code>data/</code>, so this notebook is versioned but the downloaded dataset is not.</div>


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import zipfile

import numpy as np
import pandas as pd

from jano import (
    DriftBasedRetrain,
    DriftMonitoringPolicy,
    TemporalPartitionSpec,
    TemporalSemanticsSpec,
    TrainHistoryPolicy,
    WalkForwardPolicy,
    WalkForwardRunner,
)

BTS_ZIP_URL = 'https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip'
DATA_PATH = Path("data/bts/bts_ontime_2024_01.zip")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/bts/bts_ontime_2024_01.zip")

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(BTS_ZIP_URL, DATA_PATH)

STATE_LABELS = {0: "early", 1: "on_time", 2: "delayed"}


def hhmm_to_minutes(value):
    if pd.isna(value):
        return np.nan
    value = int(value)
    return (value // 100) * 60 + (value % 100)


def load_bts_sample(path: Path, limit: int = 25_000, origin: str = "ATL") -> pd.DataFrame:
    with zipfile.ZipFile(path) as zf:
        member = zf.namelist()[0]
        with zf.open(member) as f:
            frame = pd.read_csv(
                f,
                usecols=[
                    "FlightDate",
                    "DayOfWeek",
                    "Reporting_Airline",
                    "Origin",
                    "Dest",
                    "CRSDepTime",
                    "ArrTime",
                    "ArrDelay",
                    "Distance",
                    "Cancelled",
                    "Diverted",
                ],
            )
    frame = frame.loc[(frame["Origin"] == origin) & (frame["Cancelled"] == 0) & (frame["Diverted"] == 0)].copy()
    frame = frame.head(limit).copy()
    flight_date = pd.to_datetime(frame["FlightDate"])
    dep_minutes = frame["CRSDepTime"].map(hhmm_to_minutes)
    arr_minutes = frame["ArrTime"].map(hhmm_to_minutes)
    frame["scheduled_departure_at"] = flight_date + pd.to_timedelta(dep_minutes, unit="m")
    frame["actual_arrival_at"] = flight_date + pd.to_timedelta(arr_minutes, unit="m")
    overnight = frame["actual_arrival_at"] < frame["scheduled_departure_at"]
    frame.loc[overnight, "actual_arrival_at"] += pd.Timedelta(days=1)
    frame["scheduled_dep_hour"] = (dep_minutes // 60).astype("Int64")
    delay = frame["ArrDelay"].fillna(0)
    frame["arrival_state"] = np.select([delay <= -1, delay <= 15], [0, 1], default=2).astype(int)
    frame["arrival_state_label"] = frame["arrival_state"].map(STATE_LABELS)
    return frame.sort_values("scheduled_departure_at").reset_index(drop=True)


class SimpleArrivalStateClassifier:
    def fit(self, X: pd.DataFrame, y: pd.Series):
        keys = self._keys(X)
        grouped = pd.DataFrame({"key": keys, "target": y.to_numpy()}).groupby("key")["target"]
        self.mode_by_key = grouped.agg(lambda s: int(pd.Series.mode(s).iloc[0])).to_dict()
        self.global_mode = int(pd.Series(y).mode().iloc[0])
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        keys = self._keys(X)
        return np.array([self.mode_by_key.get(key, self.global_mode) for key in keys], dtype=int)

    @staticmethod
    def _keys(X: pd.DataFrame) -> pd.Series:
        return X[["Origin", "Dest", "Reporting_Airline", "DayOfWeek", "scheduled_dep_hour"]].astype(str).agg("|".join, axis=1)


def multiclass_accuracy(y_true, y_pred) -> float:
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def ordinal_delay_cost(y_true, y_pred) -> float:
    return float(np.mean(np.abs(np.asarray(y_true, dtype=int) - np.asarray(y_pred, dtype=int))))


data/bts/bts_ontime_2024_01.zip

## 1. Load a Real Operational Slice

The raw monthly BTS file is large enough to make planning realistic. We keep only ATL departures, remove cancelled/diverted flights and retain a compact set of columns so the notebook stays fast while preserving the temporal problem.


In [ ]:
flights = load_bts_sample(DATA_PATH)
preview_columns = [
    "scheduled_departure_at",
    "actual_arrival_at",
    "Origin",
    "Dest",
    "Reporting_Airline",
    "ArrDelay",
    "arrival_state_label",
]
flights[preview_columns].head()


scheduled_departure_at,actual_arrival_at,Origin,Dest,Reporting_Airline,ArrDelay,arrival_state_label
2024-01-01 05:21,2024-01-01 07:22,ATL,FLL,NK,7.0,on_time
2024-01-01 05:35,2024-01-01 07:36,ATL,LGA,NK,-24.0,early
2024-01-01 05:36,2024-01-01 07:13,ATL,MIA,AA,-20.0,early
2024-01-01 05:47,2024-01-01 07:34,ATL,DFW,AA,14.0,on_time
2024-01-01 05:50,2024-01-01 07:34,ATL,MCO,F9,2.0,on_time


## 2. Define the Multiclass Target

Instead of a binary delayed vs not-delayed target, this notebook uses three ordered arrival states:

- `early` when `ArrDelay <= -1`
- `on_time` when `-1 < ArrDelay <= 15`
- `delayed` when `ArrDelay > 15`

That makes fold-by-fold evaluation more interesting because we can inspect not only accuracy, but also an ordinal cost that penalizes confusing `early` with `delayed` more strongly than confusing adjacent states.


In [ ]:
class_distribution = (
    flights["arrival_state_label"]
    .value_counts()
    .rename_axis("arrival_state_label")
    .reset_index(name="rows")
)
class_distribution["share"] = (class_distribution["rows"] / len(flights)).round(3)
class_distribution


arrival_state_label,rows,share
early,15082,0.603
delayed,5023,0.201
on_time,4895,0.196


## 3. Define Leakage-Aware Time Semantics

The operational timeline is `scheduled_departure_at`. Training eligibility uses `actual_arrival_at`, because the supervised target is only known once the flight has arrived. Test eligibility stays anchored on scheduled departure.


In [ ]:
semantics = TemporalSemanticsSpec(
    timeline_col="scheduled_departure_at",
    segment_time_cols={
        "train": "actual_arrival_at",
        "test": "scheduled_departure_at",
    },
)

walk_forward = WalkForwardPolicy(
    time_col=semantics,
    partition=TemporalPartitionSpec(
        layout="train_test",
        train_size="7D",
        test_size="1D",
        calendar_frequency="D",
    ),
    step="1D",
    strategy="rolling",
    max_folds=8,
)


## 4. Inspect the Plan Before Materializing Folds

A [`plan()`](https://marmurar.github.io/jano/concepts.html#planning-before-materialization) is the precomputed geometry of a simulation. It shows the temporal intent before any train/test slices are materialized or any model is trained.


In [ ]:
plan = walk_forward.plan(flights, title="ATL January 2024 walk-forward plan")
plan_df = plan.to_frame()
plan_df[["iteration", "train_start", "train_end", "test_start", "test_end", "train_rows", "test_rows"]]


iteration,train_start,train_end,test_start,test_end,train_rows,test_rows
0,2024-01-01,2024-01-08,2024-01-08,2024-01-09,5476,792
1,2024-01-02,2024-01-09,2024-01-09,2024-01-10,5594,751
2,2024-01-03,2024-01-10,2024-01-10,2024-01-11,5511,771
3,2024-01-04,2024-01-11,2024-01-11,2024-01-12,5480,806
4,2024-01-05,2024-01-12,2024-01-12,2024-01-13,5498,783
5,2024-01-06,2024-01-13,2024-01-13,2024-01-14,5461,724
6,2024-01-07,2024-01-14,2024-01-14,2024-01-15,5447,745
7,2024-01-08,2024-01-15,2024-01-15,2024-01-16,5378,801


## 5. Execute a Multiclass Model Across the Folds

`WalkForwardRunner` lets Jano take responsibility for the training and prediction loop while the splitter still owns temporal geometry. Here we compare four retraining policies over the same walk-forward plan:

- `always`
- `never`
- `periodic_2`
- `on_drift`

The custom `delay_cost` metric is the mean absolute distance between ordered classes `early -> on_time -> delayed`.


In [ ]:
feature_cols = ["Origin", "Dest", "Reporting_Airline", "DayOfWeek", "scheduled_dep_hour", "Distance"]
metrics = {"accuracy": multiclass_accuracy, "delay_cost": ordinal_delay_cost}

policy_runs = {}
for label, kwargs in {
    "always": {"retrain": "always"},
    "never": {"retrain": "never"},
    "periodic_2": {"retrain": "periodic", "retrain_interval": 2},
    "on_drift": {
        "retrain_policy": DriftBasedRetrain(
            metric="delay_cost",
            threshold=0.15,
            baseline="last_retrain",
            relative=False,
        )
    },
}.items():
    policy_runs[label] = WalkForwardRunner(
        model=SimpleArrivalStateClassifier(),
        target_col="arrival_state",
        feature_cols=feature_cols,
        metrics=metrics,
        **kwargs,
    ).run(walk_forward, flights)

comparison = []
for label, run in policy_runs.items():
    fold_frame = run.to_frame()
    comparison.append(
        {
            "policy": label,
            "folds": len(fold_frame),
            "retrain_events": int(fold_frame["retrained"].sum()),
            "mean_accuracy": round(float(fold_frame["accuracy"].mean()), 3),
            "mean_delay_cost": round(float(fold_frame["delay_cost"].mean()), 3),
        }
    )
comparison_df = pd.DataFrame(comparison).sort_values(["mean_delay_cost", "mean_accuracy"], ascending=[True, False])
comparison_df


policy,folds,retrain_events,mean_accuracy,mean_delay_cost
always,8,8,0.477,0.785
periodic_2,8,4,0.477,0.785
on_drift,8,3,0.477,0.785
never,8,1,0.475,0.793


In [ ]:
periodic_detail = policy_runs["periodic_2"].to_frame()[["fold", "retrained", "accuracy", "delay_cost"]]
periodic_detail


fold,retrained,accuracy,delay_cost
0,True,0.625,0.558
1,False,0.200,1.383
2,True,0.616,0.516
3,False,0.597,0.525
4,True,0.404,0.908
5,False,0.453,0.787
6,True,0.499,0.719
7,False,0.421,0.884


## 6. Train History Hypothesis with the Same Multiclass Target

Now the question changes: if the test day stays fixed, how much train history is actually needed to recover the best cost profile?


In [ ]:
train_history = TrainHistoryPolicy(
    semantics,
    cutoff=plan.to_frame().iloc[3]["train_end"],
    train_sizes=["2D", "4D", "6D", "7D"],
    test_size="1D",
)

train_history_result = train_history.evaluate(
    flights,
    model=SimpleArrivalStateClassifier(),
    target_col="arrival_state",
    feature_cols=feature_cols,
    metrics=metrics,
)

train_history_result.to_frame()[["train_size", "train_rows", "accuracy", "delay_cost"]]


train_size,train_rows,accuracy,delay_cost
2 days 00:00:00,1530,0.125,1.512
4 days 00:00:00,3129,0.638,0.488
6 days 00:00:00,4685,0.634,0.494
7 days 00:00:00,5480,0.597,0.525


## 7. Drift Monitoring with Ordinal Cost

This policy freezes the train window and moves test forward. The metric to watch is now more operational: accuracy may stay similar while ordinal delay cost worsens because the classifier drifts toward more severe confusions.


In [ ]:
drift_monitor = DriftMonitoringPolicy(
    semantics,
    cutoff=plan.to_frame().iloc[3]["train_end"],
    train_size="7D",
    test_size="1D",
    step="1D",
    max_windows=6,
)

drift_result = drift_monitor.evaluate(
    flights,
    model=SimpleArrivalStateClassifier(),
    target_col="arrival_state",
    feature_cols=feature_cols,
    metrics=metrics,
)

drift_result.to_frame()[["window", "accuracy", "delay_cost"]]


window,accuracy,delay_cost
0,0.597,0.525
1,0.404,0.908
2,0.453,0.787
3,0.499,0.719
4,0.421,0.884
5,0.418,0.885


## Notes

- The notebook still prioritizes temporal geometry and leakage-aware validation over model sophistication.
- `TemporalBacktestSplitter` remains available when you want manual fold iteration; the runner is an execution layer on top of that geometry.
- The multiclass target makes fold variation more informative because `delay_cost` distinguishes adjacent mistakes from severe ones.
- Replace the toy classifier with your own model pipeline while keeping the same temporal setup.
